In [83]:
import pandas as pd
import numpy as np
import re
import pickle
import random
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Downloads necessários para o NLTK (corre apenas uma vez)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

np.random.seed(42)
random.seed(42)

# Mapeamento das classes (0 a 4)
CLASS_MAP = {
    'Human': 0,
    'OpenAI': 1,
    'Google': 2,
    'Meta': 3,
    'Anthropic': 4
}

[nltk_data] Downloading package punkt to C:\Users\Gonçalo
[nltk_data]     Corais\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Gonçalo
[nltk_data]     Corais\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Gonçalo
[nltk_data]     Corais\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [84]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_and_slice(text, min_words=80, max_words=120):
    if not isinstance(text, str):
        return None
        
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    words = text.split()
    
    # Se o texto for menor que 80 palavras, ignorar
    if len(words) < min_words:
        return None
        
    slice_length = random.randint(min_words, max_words)
    
    if len(words) <= slice_length:
        fragment_words = words
    else:
        max_start_idx = len(words) - slice_length
        start_idx = random.randint(0, max_start_idx)
        fragment_words = words[start_idx : start_idx + slice_length]
        
    # Lematização
    processed = [lemmatizer.lemmatize(w.lower()) for w in fragment_words if w.isalnum()]
    
    return ' '.join(processed) if len(processed) > 3 else None

In [85]:
data_list = []
global_id_counter = 0 # Identificador único para cada texto original

# 1. Anthropic (Claude)
anthropic_files = ['datasets/claude_multi_instruct_1k.csv']
for f in anthropic_files:
    df_a = pd.read_csv(f, sep=';')
    for text in df_a['output'].dropna():
        global_id_counter += 1
        for _ in range(4):
            frag = clean_and_slice(text)
            if frag: data_list.append({'text': frag, 'label': CLASS_MAP['Anthropic'], 'original_id': global_id_counter})

# 2. Google (Gemini/Gemma)
google_files = ['datasets/gemma_1.csv', 'datasets/gemma_2.csv', 'datasets/gemini_3.csv']
for f in google_files:
    df_g = pd.read_csv(f)
    for text in df_g['rewritten_text'].dropna():
        global_id_counter += 1
        frag = clean_and_slice(text)
        if frag: data_list.append({'text': frag, 'label': CLASS_MAP['Google'], 'original_id': global_id_counter})

# 3. OpenAI & Human
df_gpt = pd.read_csv('datasets/GPT.csv')
for _, row in df_gpt.iterrows():
    global_id_counter += 1
    frag = clean_and_slice(row['abstract'])
    if frag:
        label = CLASS_MAP['OpenAI'] if row['is_ai_generated'] == 1 else CLASS_MAP['Human']
        data_list.append({'text': frag, 'label': label, 'original_id': global_id_counter})

# 4. Meta (Llama)
df_llama = pd.read_csv('datasets/llama.csv')
for text in df_llama['answer_content'].dropna():
    global_id_counter += 1
    for _ in range(15):
        frag = clean_and_slice(text)
        if frag: data_list.append({'text': frag, 'label': CLASS_MAP['Meta'], 'original_id': global_id_counter})

df_all = pd.DataFrame(data_list).drop_duplicates(subset=['text'])

In [86]:
ids_unicos = df_all[['original_id', 'label']].drop_duplicates()

ids_train, ids_temp = train_test_split(ids_unicos['original_id'], test_size=0.20, 
                                       random_state=42, stratify=ids_unicos['label'])
ids_val, ids_test = train_test_split(ids_temp, test_size=0.50, random_state=42)

df_train = df_all[df_all['original_id'].isin(ids_train)]
df_val = df_all[df_all['original_id'].isin(ids_val)]
df_test = df_all[df_all['original_id'].isin(ids_test)]

min_samples = df_train['label'].value_counts().min()
SAMPLES_PER_CLASS = min(min_samples, 2000)
df_train_bal = df_train.groupby('label').sample(n=SAMPLES_PER_CLASS, random_state=42)

print(f"Dataset pronto! Treino: {len(df_train_bal)} | Val: {len(df_val)} | Teste: {len(df_test)}")

Dataset pronto! Treino: 7415 | Val: 1435 | Teste: 1395


In [87]:
# Criar o vetorizador TF-IDF (limita as features para não esgotar a RAM com a tua rede Numpy)
vectorizer = TfidfVectorizer(max_features=1500)

# Transformar os textos na matriz tabular (features)
X_train = vectorizer.fit_transform(df_train_bal['text']).toarray()
X_val = vectorizer.transform(df_val['text']).toarray()
X_test = vectorizer.transform(df_test['text']).toarray()

# Guardar o vetorizador para o professor
with open('Subm1/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Formato dos dados de Treino (Numpy Array Tabular):", X_train.shape)
print("Formato das Labels de Treino:", df_train_bal['label'].values.shape)

# Guardar os dados processados para usar no script Numpy (Tarefa 2)
np.save('X_train.npy', X_train)
np.save('y_train.npy', df_train_bal['label'].values)
np.save('X_val.npy', X_val)
np.save('y_val.npy', df_val['label'].values)
np.save('X_test.npy', X_test)
np.save('y_test.npy', df_test['label'].values)

print("Dataset tabular criado e guardado com sucesso!")

Formato dos dados de Treino (Numpy Array Tabular): (7415, 1500)
Formato das Labels de Treino: (7415,)
Dataset tabular criado e guardado com sucesso!
